
# Free Throw Success Classification (Beginner-Friendly)

This project uses the `combined.csv` dataset (NBA per-game player-season stats from Basketball-Reference) to build **a classification model**.

## Important: What is the target here?
This dataset is **not shot-by-shot free throw attempts** (it does not have a `made/missed` outcome per free throw). Instead it has season-level *rates* like `FT%`.

So we define a clean binary target:

- `target = 1` if a player's **season FT%** is **greater than or equal to the league average FT%** for that same season.
- `target = 0` otherwise.

This gives us a meaningful and well-defined classification task: **is this player an above-average free throw shooter (for that season)?**

We also take care to avoid leakage:
- We **drop `FT%` from the features** (because it's used to build the target).
- We avoid directly using anything that is a trivial re-expression of `FT%`.


# 02. Cleaning + Preprocessing (Leakage-Safe)

We will:
- remove `League Average` rows
- keep one row per player-season (prefer `2TM` / `3TM` summary)
- create `target`
- split train/test with stratification
- build a scikit-learn `ColumnTransformer` pipeline for encoding + scaling
- save `X_train`, `X_test`, `y_train`, `y_test` to disk for reuse in all model notebooks

Saving splits avoids accidental differences between notebooks.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

DATA_PATH = 'combined.csv'
ARTIFACT_DIR = Path('artifacts')
ARTIFACT_DIR.mkdir(exist_ok=True)

df_raw = pd.read_csv(DATA_PATH)
print('Raw shape:', df_raw.shape)

Raw shape: (26292, 32)


## 1. Create a Clean Player-Season Table

In [2]:
df = df_raw[df_raw['Player'] != 'League Average'].copy()

team_is_tm = df['Team'].astype(str).str.match(r'^\d+TM$')
rows = []
for (season, player), grp in df.groupby(['Season','Player'], sort=False):
    tm = grp[team_is_tm.loc[grp.index]]
    rows.append(tm.iloc[[0]] if len(tm) else grp.iloc[[0]])

df = pd.concat(rows, ignore_index=True)
print('After player-season de-dup:', df.shape)

league = df_raw[df_raw['Player']=='League Average'][['Season','FT%']].rename(columns={'FT%':'league_ft_pct'})
df = df.merge(league, on='Season', how='left')

# Drop rows that cannot form target
df = df.dropna(subset=['FT%','league_ft_pct']).copy()

# Target: above/equal league average FT%
df['target'] = (df['FT%'] >= df['league_ft_pct']).astype(int)

print('After target creation:', df.shape)
print(df['target'].value_counts(normalize=True))

After player-season de-dup: (21276, 32)
After target creation: (20585, 34)
target
0    0.521691
1    0.478309
Name: proportion, dtype: float64


## 2. Decide What to Drop (Leakage / IDs / Redundant)

We drop:
- `FT%` (used to create target)
- `league_ft_pct` (directly used in target)
- `Player` (ID-like; model would memorize players)
- `Rk` (rank index, not a real feature)
- `Awards` (90% missing and also leaks reputation rather than skill)

We **keep** `FT` and `FTA` because they reflect volume and opportunities. They do not directly reveal `FT%` without combining them, and many models can still use them sensibly.


In [3]:
DROP_COLS = ['target','FT%','league_ft_pct','Player','Rk','Awards']

X = df.drop(columns=DROP_COLS)
y = df['target']

print('Features shape:', X.shape)
print('Target shape:', y.shape)
print('Columns:', list(X.columns))

Features shape: (20585, 28)
Target shape: (20585,)
Columns: ['Age', 'Team', 'Pos', 'G', 'GS', 'MP', 'FG', 'FGA', 'FG%', 'FT', 'FTA', 'ORB', 'DRB', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS', 'Season', '3P', '3PA', '3P%', '2P', '2PA', '2P%', 'eFG%']


## 3. Basic Data Quality Checks

In [4]:
print('Missing fraction (top 15):')
print(X.isna().mean().sort_values(ascending=False).head(15))

print('Categorical columns:', X.select_dtypes(include='object').columns.tolist())
print('Numeric columns:', X.select_dtypes(include='number').columns.tolist())

Missing fraction (top 15):
3P%     0.184746
GS      0.071363
2P%     0.064562
eFG%    0.063930
3P      0.063541
2PA     0.063541
2P      0.063541
3PA     0.063541
TOV     0.036143
FG%     0.000389
Age     0.000000
Team    0.000000
ORB     0.000000
FTA     0.000000
FT      0.000000
dtype: float64
Categorical columns: ['Team', 'Pos']
Numeric columns: ['Age', 'G', 'GS', 'MP', 'FG', 'FGA', 'FG%', 'FT', 'FTA', 'ORB', 'DRB', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS', 'Season', '3P', '3PA', '3P%', '2P', '2PA', '2P%', 'eFG%']


## 4. Train/Test Split (Stratified)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train:', X_train.shape, ' Test:', X_test.shape)
print('Train positive rate:', y_train.mean())
print('Test positive rate:', y_test.mean())

Train: (16468, 28)  Test: (4117, 28)
Train positive rate: 0.47832159339324753
Test positive rate: 0.4782608695652174


## 5. Preprocessing Pipeline

- Categorical: impute missing with most-frequent, then one-hot encode.
- Numeric: impute missing with median, then scale.

Scaling is especially important for KNN / SVC / Logistic Regression / Neural Nets.

In [6]:
cat_cols = X_train.select_dtypes(include='object').columns.tolist()
num_cols = X_train.select_dtypes(include='number').columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocess = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols),
    ]
)

preprocess

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


## 6. Save Split + Column Lists

We save the raw `X_train/X_test` (before transformation) and `y` so each model notebook can load the exact same split.


In [7]:
import joblib

joblib.dump({
    'X_train': X_train,
    'X_test': X_test,
    'y_train': y_train,
    'y_test': y_test,
    'preprocess': preprocess,
    'cat_cols': cat_cols,
    'num_cols': num_cols,
}, ARTIFACT_DIR / 'data_split_and_preprocess.joblib')

print('Saved:', ARTIFACT_DIR / 'data_split_and_preprocess.joblib')

Saved: artifacts/data_split_and_preprocess.joblib
